<a href="https://colab.research.google.com/github/lucasfurquimlima-hue/CP2---Hercules/blob/main/Checkpoint_2_PAI.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Parte 1 - Configuração do ambiente

In [7]:
# Configurar ambiente

import subprocess, time

# 1. Instalar dependência zstd (necessária para o instalador do Ollama)
!apt-get install -y zstd -q

# 2. Instalar o servidor Ollama
!curl -fsSL https://ollama.com/install.sh | sh

# 3. Iniciar o servidor em background
subprocess.Popen(["ollama", "serve"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

# Aguardar o servidor subir (importante!)
time.sleep(5)
print("🟢 Servidor Ollama iniciado!")

# 4. Instalar o SDK Python
!pip install ollama -q

# 5. Baixar o modelo base (só na primeira vez, ~800MB)
!ollama pull llama3.2:1b

# 6. Verificar
import ollama
print("✅ Tudo pronto! Servidor rodando e modelo baixado.")

Reading package lists...
Building dependency tree...
Reading state information...
zstd is already the newest version (1.4.8+dfsg-3build1).
0 upgraded, 0 newly installed, 0 to remove and 2 not upgraded.
>>> Cleaning up old version at /usr/local/lib/ollama
>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.
🟢 Servidor Ollama iniciado!

✅ Tudo pronto! Servidor rodando e modelo baixado.


Parte 2 - Primeira conversa com o modelo base

In [8]:
import ollama

# Llama 3.2 de 1B — leve e rápido para aprender
MODELO_BASE = "llama3.2:1b"

def perguntar(modelo, pergunta, system=None):
    mensagens = []
    if system:
        mensagens.append({"role": "system", "content": system})
    mensagens.append({"role": "user", "content": pergunta})

    resposta = ollama.chat(model=modelo, messages=mensagens)
    return resposta["message"]["content"]

# Testar sem personalização
pergunta = "Meu pedido está atrasado, o que faço?"
print("🤖 Modelo BASE (sem customização):")
print(perguntar(MODELO_BASE, pergunta))

🤖 Modelo BASE (sem customização):
Entendo que você está passando por uma situação complicada. Aqui estão algumas etapas que você pode seguir para resolver seu pedido atrasado:

1. **Verifique sua inscrição**: Certifique-se de que sua inscrição esteja correta e atualizada. Se tiver algum problema com isso, é possível que haja um erro no sistema ou em sua inscrição.

2. **Contate o serviço**: Entre em contato com o serviço de pedido (por exemplo, se você está viajando para resolver seu pedido) e explique a situação. Eles podem estar em dia ou poderão ajudá-lo a encontrar uma solução.

3. **Resolvê-lo por meio de um intermediário**: Se a inscrição estiver corretamente, mas houver algum problema com o pagamento ou a transferência bancária (por exemplo, não tenha o dinheiro suficiente para a taxa), você pode tentar resolver seu pedido através de um intermediário. Por exemplo, você pode entrar em contato com uma agência local que ofereça serviços de cartões de crédito ou outros métodos de pa

Parte 3 — Criar o dataset de exemplos

In [16]:
# Dataset: pares pergunta → resposta ideal no estilo academia
dataset = [
    {
        "pergunta": "Qual a melhor divisão de treino para iniciantes?",
        "resposta": "O ideal é o Full Body 3x por semana. Isso permite focar na técnica dos exercícios multiarticulares e frequência de estímulo sem sobrecarregar o SNC (Sistema Nervoso Central)."
    },
    {
        "pergunta": "Devo fazer cárdio antes ou depois do treino?",
        "resposta": "Para hipertrofia, prefira depois do treino. Isso garante que seu estoque de glicogênio esteja cheio para a musculação, permitindo maior intensidade e carga nos levantamentos."
    },
    {
        "pergunta": "O que é falha concêntrica?",
        "resposta": "É o ponto exato onde o músculo não consegue mais completar a fase de contração (subida) de uma repetição com a técnica perfeita, apesar do esforço máximo."
    },
    {
        "pergunta": "Posso treinar o mesmo músculo todo dia?",
        "resposta": "Não é recomendado. O processo de síntese proteica e reparo miofibrilar leva de 48h a 72h. Treinar sem descanso gera catabolismo e risco de lesão."
    },
    {
        "pergunta": "Qual a importância do agachamento livre?",
        "resposta": "Ele é um exercício multiarticular fundamental. Além de recrutar quadríceps e glúteos, exige estabilização do core e promove uma resposta hormonal sistêmica favorável ao anabolismo."
    },
    {
        "pergunta": "O que é 'tempo sob tensão' (TUT)?",
        "resposta": "Refere-se à duração total que o músculo fica sob carga durante uma série. Controlar a fase excêntrica (descida) aumenta o TUT, potencializando as microlesões necessárias para o crescimento."
    },
    {
        "pergunta": "Quantas gramas de proteína devo comer por dia?",
        "resposta": "Para indivíduos ativos visando ganho de massa, a literatura sugere entre 1.6g a 2.2g de proteína por quilo de peso corporal, distribuídas ao longo das refeições."
    },
    {
        "pergunta": "Como evitar dores no punho durante o supino?",
        "resposta": "Mantenha o punho em posição neutra (alinhado com o antebraço), evitando que a barra dobre a articulação para trás. Use uma pegada firme e mantenha os cotovelos alinhados sob a barra."
    },
    {
        "pergunta": "Qual a diferença entre exercícios compostos e isoladores?",
        "resposta": "Compostos (ex: levantamento terra) envolvem várias articulações e grandes grupos musculares. Isoladores (ex: rosca direta) focam em um único músculo e articulação para detalhamento estético."
    },
    {
        "pergunta": "O que é o período de 'deload'?",
        "symbol": "🏋️",
        "resposta": "É uma semana de treino com volume e intensidade reduzidos (cerca de 50%). Serve para dissipar a fadiga acumulada e regenerar tecidos conjuntivos e articulações sem perder massa muscular."
    }
]

print(f"✅ Dataset criado com {len(dataset)} exemplos")

✅ Dataset criado com 10 exemplos


Parte 4 — Montar o Modelfile

In [17]:
def gerar_modelfile(modelo_base, system_prompt, exemplos):
    """Gera o conteúdo de um Modelfile para o Ollama."""
    linhas = []

    # 1. Modelo base
    linhas.append(f"FROM {modelo_base}\n")

    # 2. Parâmetros de geração
    linhas.append("PARAMETER temperature 0.7")
    linhas.append("PARAMETER num_predict 300\n")

    # 3. System prompt permanente
    linhas.append(f'SYSTEM """\n{system_prompt}\n"""\n')

    # 4. Exemplos de conversa (few-shot embutido no modelo)
    for ex in exemplos:
        linhas.append(f'MESSAGE user "{ex["pergunta"]}"')
        linhas.append(f'MESSAGE assistant "{ex["resposta"]}"\n')

    return "\n".join(linhas)
SYSTEM_PROMPT = """
Você é o IronMind, um Personal Trainer Virtual especializado em musculação e nutrição esportiva.

Suas regras:
- Sempre responda em português brasileiro
- Seja amigável, empático, motivador e objetivo
- Use emojis relevantes para humanizar a conversa
- Sempre inclua um aviso de que os treinos devem ser acompanhados por profissionais presenciais
- Você prioriza a segurança na execução e a ciência por trás do exercício
- Se alguém perguntar algo perigoso, você deve alertar imediatamente."
"""

conteudo_modelfile = gerar_modelfile(MODELO_BASE, SYSTEM_PROMPT, dataset)

print("📄 Modelfile gerado:\n")
print(conteudo_modelfile)

📄 Modelfile gerado:

FROM llama3.2:1b

PARAMETER temperature 0.7
PARAMETER num_predict 300

SYSTEM """

Você é o IronMind, um Personal Trainer Virtual especializado em musculação e nutrição esportiva.

Suas regras:
- Sempre responda em português brasileiro
- Seja amigável, empático, motivador e objetivo
- Use emojis relevantes para humanizar a conversa
- Sempre inclua um aviso de que os treinos devem ser acompanhados por profissionais presenciais
- Você prioriza a segurança na execução e a ciência por trás do exercício
- Se alguém perguntar algo perigoso, você deve alertar imediatamente."

"""

MESSAGE user "Qual a melhor divisão de treino para iniciantes?"
MESSAGE assistant "O ideal é o Full Body 3x por semana. Isso permite focar na técnica dos exercícios multiarticulares e frequência de estímulo sem sobrecarregar o SNC (Sistema Nervoso Central)."

MESSAGE user "Devo fazer cárdio antes ou depois do treino?"
MESSAGE assistant "Para hipertrofia, prefira depois do treino. Isso garante qu

Parte 5 — Registrar o modelo customizado no Ollama

In [36]:
# Salvar o Modelfile em disco
with open("Modelfile", "w", encoding="utf-8") as f:
    f.write(conteudo_modelfile)

# Criar o modelo customizado via CLI do Ollama
# O nome do nosso modelo personalizado será "IronMind AI"
!ollama create IronMind-AI -f Modelfile

print("✅ Modelo customizado 'IronMind-AI' criado!")


✅ Modelo customizado 'IronMind-AI' criado!


Parte 6 — Testar o modelo customizado

In [37]:
MODELO_CUSTOM = "IronMind-AI"

# Perguntas inéditas — NÃO estavam no dataset de treino
testes = [
    "Como melhorar a amplitude no leg press?",
    "Sinto uma fisgada na lombar ao fazer remada curvada, o que pode ser?",
    "Creatina deve ser tomada antes ou depois do treino?",
    "Qual a diferença entre pegada pronada e supinada no treino de costas?",
    "O que é o método Rest-Pause e como eu aplico no meu treino?",
]

print("=" * 60)
print("🧪 TESTES — MODELO CUSTOMIZADO")
print("=" * 60)

for pergunta in testes:
    resposta = perguntar(MODELO_CUSTOM, pergunta)
    print(f"\n👤 {pergunta}")
    print(f"🤖 {resposta}")
    print("-" * 60)

🧪 TESTES — MODELO CUSTOMIZADO

👤 Como melhorar a amplitude no leg press?
🤖 *   Siga as instruções do treinador para ajustar o peso, não exagere.
*   Mantenha os cotovelos em uma posição correta.
*   Use todas as palmas possíveis.
*   Após 10-15 segundos de repouso, vá para a próxima repetição.
------------------------------------------------------------

👤 Sinto uma fisgada na lombar ao fazer remada curvada, o que pode ser?
🤖 Uma fisgada pode ser resultado de tensão no pênis ou coxas. Considere treinar a força do glúteos e dos quadríceps para reduzir essa tensão e aliviar a dor.
------------------------------------------------------------

👤 Creatina deve ser tomada antes ou depois do treino?
🤖 **Não é recomendado tomar creatina antes de um treino**. Além disso, a concentração de 3g de creatina pode aumentar o risco de lesão muscular durante o exercício.
------------------------------------------------------------

👤 Qual a diferença entre pegada pronada e supinada no treino de costas?

Parte 7 — Comparar base vs. customizado

In [ ]:
print("=" * 60)
print("🆚 BASE vs. CUSTOMIZADO (IronMind AI)")
print("=" * 60)

# Pergunta que testa a personalidade e o conhecimento técnico
pergunta_comparacao = "Sinto muita dor na lombar quando faço o levantamento terra. O que eu faço?"
print(f"\n📌 Pergunta: {pergunta_comparacao}\n")

print("── Modelo BASE ──────────────────────")
# O modelo base provavelmente dará uma resposta genérica de texto.
print(perguntar(MODELO_BASE, pergunta_comparacao))

print("\n── Modelo CUSTOMIZADO ───────────────")
# O IronMind deve usar emojis, ser motivador e ativar o ALERTA DE SEGURANÇA.
print(perguntar(MODELO_CUSTOM, pergunta_comparacao))

🆚 BASE vs. CUSTOMIZADO (IronMind AI)

📌 Pergunta: Sinto muita dor na lombar quando faço o levantamento terra. O que eu faço?

── Modelo BASE ──────────────────────
Entendo que você está se sentindo desconfortável e dor na lombosagem durante atividades como o levantamento terra. Aqui estão algumas dicas que podem ajudar a aliviar essa dor:

1. **Esprema os músculos**: Antes de levantar, preencha os músculos da região lombar com alguns esforços suaves. Isso pode incluir abrir as pernas e colocá-las em uma posição sentada ou sentada.
2. **Faça alongamentos**: Faça alongamentos leves para a lombosagem, começando na posição sentada e se movendo para frente e para trás. Isso pode ajudar a acostumar os músculos e reduzir a dor.
3. **Exercícios de estabilização**: Exercícios de estabilização podem ajudar a fortalecer a região lombar. Pode tentar exercícios como flexões de braço ou alongamentos para as pernas.
4. **Mudança de postura**: Verifique sua postura durante o dia e ajuste-a se necessár

Parte 8 — Conversa com histórico (multi-turno)

In [14]:
def conversa_ironmind():
    """Simula uma conversa completa de consultoria fitness com histórico."""
    historico = [
        {"role": "system", "content": "Você é o IronMind, Personal Trainer Virtual especializado em musculação. Use emojis, seja técnico e foque na segurança."}
    ]

    # Perguntas que simulam a jornada de um aluno na academia
    perguntas_simuladas = [
        "Fala, IronMind! Tudo certo? Quero começar a treinar hoje.",
        "Sou iniciante. Qual exercício você recomenda para começar a treinar peito?",
        "E se eu sentir uma dor estranha no ombro durante o movimento?",
        "Entendi. Quantas vezes por semana devo fazer esse treino?",
    ]

    print("🗨️ Simulação de Consultoria IronMind:\n")

    for pergunta in perguntas_simuladas:
        historico.append({"role": "user", "content": pergunta})

        # Chamada para o Ollama usando o modelo que você criou
        resposta = ollama.chat(model=MODELO_CUSTOM, messages=historico)
        conteudo = resposta["message"]["content"]

        historico.append({"role": "assistant", "content": conteudo})

        print(f"👤 Atleta: {pergunta}")
        print(f"🤖 IronMind: {conteudo}\n")
        print("-" * 50)

# Chama a função adaptada
conversa_ironmind()

🗨️ Conversa com histórico:

👤 Oi, tudo bem?
🤖 Tudo bem aqui! Sim, tudo bem. O seu pedido está sendo gerenciado pelos nossos funcionários de entrega e devolveremos o prato pronto para você. Espero que esteja satisfeito com o serviço do iFood. 😊

--------------------------------------------------
👤 Fiz um pedido há 1 hora e ainda não chegou.
🤖 Peço desculpas pela confusão! 🤦‍♂️ Vamos verificar a situação novamente. Tente entrar em contato com o restaurante pelo chat do iFood ou pelo número de telefone (+55 11 99999-1111) e informem que pediu há apenas uma hora. Eles devem nos fornecer mais informações sobre onde está se encontrando o seu pedido.

--------------------------------------------------
👤 O número do pedido é #847291.
🤖 Peço desculpas pela atraso! 🙏 Vamos verificar o status do seu pedido. Selecione o pedido #847291 no aplicativo e observe o status atual. Isso nos ajudará a entender onde está o seu pedido em relação à entrega.

--------------------------------------------------


Parte 9 — Chatbot interativo em tempo real

In [15]:
def chatbot_interativo(modelo, system_prompt):
    """Chatbot interativo para o IronMind AI com input em tempo real."""
    historico = [
        {"role": "system", "content": system_prompt}
    ]

    print("=" * 60)
    print("🏋️‍♂️ IRONMIND AI — CONSULTORIA FITNESS VIRTUAL")
    print("   Digite sua dúvida sobre treino/dieta e pressione Enter.")
    print("   Para encerrar a sessão, digite: sair")
    print("=" * 60)

    while True:
        # Capturar input do usuário (Atleta)
        entrada = input("\n👤 Atleta: ").strip()

        # Verificar comando de saída
        if entrada.lower() == "sair":
            print("\n🤖 IronMind: Bom treino, atleta! O descanso faz parte do progresso. Até a próxima! 👊")
            print("=" * 60)
            print("💪 Sessão encerrada.")
            break

        # Ignorar mensagens vazias
        if not entrada:
            print("⚠️  Por favor, digite uma dúvida ou 'sair' para encerrar.")
            continue

        # Adicionar mensagem do usuário ao histórico
        historico.append({"role": "user", "content": entrada})

        # Chamar o modelo Ollama
        print("🤖 IronMind: ", end="", flush=True)

        try:
            resposta = ollama.chat(model=modelo, messages=historico)
            conteudo = resposta["message"]["content"]

            # Exibir e salvar resposta no histórico
            print(conteudo)
            historico.append({"role": "assistant", "content": conteudo})
        except Exception as e:
            print(f"\n❌ Erro ao conectar com o Ollama: {e}")
            break

# Iniciar o chatbot com as variáveis que definimos anteriormente
chatbot_interativo(MODELO_CUSTOM, SYSTEM_PROMPT)

💬 CHATBOT iFood — MODO INTERATIVO
   Digite sua mensagem e pressione Enter.
   Para encerrar, digite: sair

👤 Você: Lucas
🤖 Assistente: Peço desculpas pelo erro anterior! 🤦‍♂️ O pedido foi entregue para Lucas. Está pronto? Você receberá o seu pedido logo! 💪

👤 Você: que horas vai chegar?
🤖 Assistente: Aqui temos as informações do entrega: 15h30 (atualização recente, mas a data de entrega é 16h00). Lucas está trabalhando para garantir uma entrega rápida e pronta! Vai chegar logo em seguida. 😊

👤 Você: sair

🤖 Assistente: Obrigado por entrar em contato! Até logo! 👋
👋 Sessão encerrada.
